In [ ]:
import tensorflow as tf
from keras import layers,models
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
IMG_HEIGHT = 32
IMG_WIDTH = 32
BATCH_SIZE = 32
NUM_CLASSES = 43
train_df = tf.keras.utils.image_dataset_from_directory(
    r"D:/datasets/Train",
    validation_split = 0.2,
    subset = "training",
    seed = 123,
    image_size = (IMG_HEIGHT,IMG_WIDTH),
    batch_size = BATCH_SIZE
)
class_names = train_df.class_names

val_ds = tf.keras.utils.image_dataset_from_directory(
    r"D:/datasets/Train",
    validation_split = 0.2,
    subset = "validation",
    seed = 123,
    image_size = (IMG_HEIGHT,IMG_WIDTH),
    batch_size = BATCH_SIZE
)
AUTOTUNE = tf.data.AUTOTUNE
train_df = train_df.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)


In [ ]:
def plot_history(h,title):
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.plot(h.history["accuracy"],label = "train")
    plt.plot(h.history["val_accuracy"],label ="val")
    plt.title(title + " accuracy")
    plt.subplot(1,2,2)
    plt.plot(h.history["loss"],label = "train")
    plt.plot(h.history["val_loss"],label = "val")
    plt.title(title + " loss")
    plt.legend()
    plt.show()

In [ ]:
model = models.Sequential([
    layers.Rescaling(1./255,input_shape = (IMG_HEIGHT,IMG_WIDTH,3)),
    layers.Conv2D(32,(3,3),activation="relu"),
    layers.MaxPool2D((2,2)),
    layers.Conv2D(64,(3,3),activation="relu"),
    layers.MaxPool2D((2,2)),
    layers.Flatten(),
    layers.Dense(128,activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES,activation="softmax")
])

model.compile(
    optimizer = "adam",
    loss = "sparse_categorical_crossentropy",
    metrics = ["accuracy"]
)
model.summary()

history = model.fit(
    train_df,
    shuffle=False,
    validation_data=val_ds,
    epochs = 10
)

In [ ]:
plot_history(history,"before")

## overfitting analysis (before)
training accuracy reached about 98.5% and validation accuracy about 99%.the two curves are very close.
so the model doesnt overfit and validation accuracy is higher than training accuracy

In [ ]:
augment = tf.keras.Sequential([
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1,0.1),
])
augment = tf.keras.Sequential([
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
])

model2 = models.Sequential([
    layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    augment,
    layers.Rescaling(1./255),
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation="softmax"),
])

model2.compile(optimizer="adam",
               loss="sparse_categorical_crossentropy",
               metrics=["accuracy"])

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True)

history2 = model2.fit(train_df, shuffle=False,
                      validation_data=val_ds,
                      epochs=20, callbacks=[early_stop])

In [ ]:
plot_history(history2, "After")

In [ ]:
test_csv = pd.read_csv(r"D:/datasets/Test.csv")
x_test = []
for path in test_csv["Path"]:
    img = tf.keras.utils.load_img(
        r"D:/datasets/" + path,
        target_size = (IMG_HEIGHT,IMG_WIDTH)
    )
    x_test.append(tf.keras.utils.img_to_array(img))
    
    
x_test = np.array(x_test)
y_test = np.array([class_names.index(str(c))for c in test_csv["ClassId"]])
test_loss,test_accuracy = model.evaluate(x_test,y_test,batch_size=BATCH_SIZE)
print(f"test accuracy: {test_accuracy*100:.2f}")

In [ ]:
loss2, acc2 = model2.evaluate(x_test, y_test)
print(f"model  (Before): {test_accuracy*100:.2f}%")
print(f"model2 (After) : {acc2*100:.2f}%")

## Model vs Model2 on the Test Set
Model reached 94.13% test accuracy, while Model2 reached 90.86%, so
Model2 performed worse by about 3.3 points. The augmentation
(rotation, zoom, translation) is probably too aggressive for 32x32
images, where small details such as speed-limit digits are easily
distorted, and EarlyStopping was driven by a validation set that is
optimistic because of near-duplicate frames. Test accuracy is lower
than validation accuracy for both models, which confirms that the
validation score was too optimistic.

In [ ]:
y_pred = np.argmax(model.predict(x_test),axis=1)
print(classification_report(y_test,y_pred))

## Why Apply Fixes if the Model Did Not Overfit?
The first model showed no significant overfitting: training and
validation curves were close. I still applied augmentation (without
flips), Dropout and EarlyStopping to improve generalization to new
conditions such as different angles, zoom and lighting, and to measure
whether it makes a difference.

## Why Validation Accuracy Looks Too Good
GTSRB images come from video tracks: each physical sign appears in about
30 near-duplicate consecutive frames. Because I split train/validation
randomly, near-identical frames of the same sign appear in both sets, so
validation accuracy is optimistic. The test set contains different
signs, so test accuracy is the more reliable number.

## Architecture and Hyperparameters
The model uses two Conv2D blocks (32 and 64 filters, 3x3, ReLU) with
MaxPooling to extract features, followed by Dense(128) with Dropout(0.3)
and a Softmax output of 43 classes. I used Adam with
sparse_categorical_crossentropy, batch size 32 and 32x32 inputs;
augmentation without flips and EarlyStopping (patience=3) reduce
overfitting.